# Optimization and Probabilistic Modeling - Complete Tutorial

Welcome to the comprehensive tutorial for the **Optimization and Probabilistic Modeling** package!

This notebook demonstrates all major features of the package including:
- ✅ Optimization algorithms (SGD, Adam, RMSprop)
- ✅ MCMC sampling (Metropolis-Hastings, MCMC Hammer)
- ✅ Probability distributions (Continuous & Discrete)
- ✅ PCA/PPCA with missing data handling
- ✅ ODE solvers for dynamical systems
- ✅ Likelihood functions and model fitting

**Author:** Mohammad Reza Askari  
**Package Version:** 0.1.0  
**Date:** November 2025

## 1. Setup and Imports

First, let's import all necessary libraries and set up our environment.

In [ ]:
# Core imports
import numpy as np
import jax.numpy as jnp
import jax
from jax import random
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Import package modules
from optimization_function import Optimizer
from Models import PPCA, PCA
from sampler_algorithms import MetropolisHastings, MCMCHammer, ModelParallelizer
from Continuous import Normal, Uniform, Beta, Gamma, StudentT
from Discrete import Binomial, Poisson
from Ordinary_Differential_Equation_Solvers import ODESolvers

print("✅ All imports successful!")
print(f"JAX version: {jax.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Optimization Algorithms

The package provides several optimization algorithms for gradient-based learning:
- **SGD** (Stochastic Gradient Descent): Classic optimizer with momentum support
- **Adam**: Adaptive learning rate optimizer with bias correction
- **RMSprop**: Root Mean Square Propagation optimizer

Let's demonstrate optimization by minimizing the Rosenbrock function: $f(x,y) = (a-x)^2 + b(y-x^2)^2$

In [ ]:
# Define Rosenbrock function and its gradient
def rosenbrock(params, a=1, b=100):
    x, y = params
    return (a - x)**2 + b * (y - x**2)**2

def rosenbrock_grad(params, a=1, b=100):
    x, y = params
    dx = -2*(a - x) - 4*b*x*(y - x**2)
    dy = 2*b*(y - x**2)
    return np.array([dx, dy])

# Initialize optimizers
learning_rate = 0.001
initial_params = np.array([-1.0, 1.0])

optimizers = {
    'SGD': Optimizer(initial_params.copy(), learning_rate=learning_rate, method='SGD'),
    'Adam': Optimizer(initial_params.copy(), learning_rate=learning_rate, method='ADAM'),
    'RMSprop': Optimizer(initial_params.copy(), learning_rate=0.01, method='RMSprop')
}

# Training loop
n_iterations = 1000
history = {name: {'params': [], 'loss': []} for name in optimizers.keys()}

for name, opt in optimizers.items():
    params = initial_params.copy()
    for i in range(n_iterations):
        loss = rosenbrock(params)
        grad = rosenbrock_grad(params)
        params = opt.fit(grad)
        
        if i % 100 == 0:
            history[name]['params'].append(params.copy())
            history[name]['loss'].append(loss)

print("Final Results:")
for name, opt in optimizers.items():
    final_loss = history[name]['loss'][-1]
    print(f"{name:8s}: Final loss = {final_loss:.6f}")

In [ ]:
# Visualize optimization trajectories
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot loss curves
for name, hist in history.items():
    iterations = np.arange(0, n_iterations, 100)
    ax1.plot(iterations, hist['loss'], label=name, linewidth=2)

ax1.set_xlabel('Iteration')
ax1.set_ylabel('Loss')
ax1.set_title('Optimization Convergence')
ax1.set_yscale('log')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot optimization path on contour plot
x_range = np.linspace(-2, 2, 100)
y_range = np.linspace(-1, 3, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = np.array([[rosenbrock([x, y]) for x in x_range] for y in y_range])

ax2.contour(X, Y, Z, levels=np.logspace(-0.5, 3.5, 20), cmap='viridis', alpha=0.6)
ax2.plot(1, 1, 'r*', markersize=15, label='Optimum')

for name, hist in history.items():
    params_array = np.array(hist['params'])
    ax2.plot(params_array[:, 0], params_array[:, 1], 'o-', label=name, linewidth=2, markersize=6)

ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Optimization Trajectories on Rosenbrock Function')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. MCMC Sampling

The package provides Markov Chain Monte Carlo (MCMC) algorithms for Bayesian inference:
- **Metropolis-Hastings**: Classic MCMC with customizable proposal distributions
- **MCMC Hammer**: Ensemble sampler with stretch moves for efficient sampling

Let's sample from a bivariate normal distribution and compare the results.

In [ ]:
# Define target distribution: bivariate normal
mean = np.array([2.0, -1.0])
cov = np.array([[1.0, 0.7], [0.7, 1.0]])
cov_inv = np.linalg.inv(cov)
log_det_cov = np.linalg.slogdet(cov)[1]

def log_posterior(theta):
    """Log probability of bivariate normal"""
    diff = theta - mean
    return -0.5 * (diff @ cov_inv @ diff + log_det_cov + 2 * np.log(2 * np.pi))

# Metropolis-Hastings Sampler
n_samples = 5000
initial_theta = np.array([0.0, 0.0])

mh_sampler = MetropolisHastings(
    initial_theta=initial_theta,
    log_posterior=log_posterior,
    proposal_std=0.5
)

print("Running Metropolis-Hastings...")
mh_samples = []
for _ in tqdm(range(n_samples), desc="MH Sampling"):
    sample, accepted = mh_sampler.sample()
    mh_samples.append(sample)

mh_samples = np.array(mh_samples)
acceptance_rate = mh_sampler.acceptance_rate
print(f"MH Acceptance Rate: {acceptance_rate:.2%}")

# MCMC Hammer (Ensemble Sampler)
n_walkers = 32
n_steps = 200

# Initialize walkers
initial_positions = np.random.randn(n_walkers, 2) * 0.5

print("\nRunning MCMC Hammer (Ensemble Sampler)...")
hammer = MCMCHammer(
    log_posterior=log_posterior,
    initial_positions=initial_positions,
    n_walkers=n_walkers
)

hammer_samples = []
for _ in tqdm(range(n_steps), desc="MCMC Hammer"):
    samples, _ = hammer.sample()
    hammer_samples.append(samples)

# Flatten walker chains
hammer_samples = np.array(hammer_samples).reshape(-1, 2)
print(f"Total MCMC Hammer samples: {len(hammer_samples)}")

In [ ]:
# Visualize MCMC results
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Burn-in removal
burn_in = 1000
mh_samples_burned = mh_samples[burn_in:]

# True distribution contours
x_range = np.linspace(-2, 6, 100)
y_range = np.linspace(-5, 3, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = np.zeros_like(X)
for i in range(len(x_range)):
    for j in range(len(y_range)):
        Z[j, i] = np.exp(log_posterior(np.array([X[j, i], Y[j, i]])))

# Plot 1: MH Trace plots
axes[0, 0].plot(mh_samples[:, 0], alpha=0.7, linewidth=0.5)
axes[0, 0].axvline(burn_in, color='r', linestyle='--', label='Burn-in')
axes[0, 0].axhline(mean[0], color='g', linestyle='--', label='True mean')
axes[0, 0].set_title('MH Trace: Parameter 1')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].legend()

# Plot 2: MH Samples
axes[0, 1].contour(X, Y, Z, levels=10, alpha=0.5)
axes[0, 1].scatter(mh_samples_burned[:, 0], mh_samples_burned[:, 1], 
                   alpha=0.3, s=5, label='MH Samples')
axes[0, 1].plot(mean[0], mean[1], 'r*', markersize=15, label='True Mean')
axes[0, 1].set_title('Metropolis-Hastings Samples')
axes[0, 1].set_xlabel('Parameter 1')
axes[0, 1].set_ylabel('Parameter 2')
axes[0, 1].legend()

# Plot 3: MH Marginal distributions
axes[0, 2].hist(mh_samples_burned[:, 0], bins=50, density=True, alpha=0.7, label='Samples')
x_theory = np.linspace(mh_samples_burned[:, 0].min(), mh_samples_burned[:, 0].max(), 100)
y_theory = np.exp(-0.5 * ((x_theory - mean[0]) / np.sqrt(cov[0, 0]))**2) / np.sqrt(2 * np.pi * cov[0, 0])
axes[0, 2].plot(x_theory, y_theory, 'r-', linewidth=2, label='True PDF')
axes[0, 2].set_title('MH Marginal: Parameter 1')
axes[0, 2].legend()

# Plot 4: Hammer Trace (first walker)
walker_0_samples = np.array(hammer_samples).reshape(n_steps, n_walkers, 2)[:, 0, :]
axes[1, 0].plot(walker_0_samples[:, 0], alpha=0.7, linewidth=0.5)
axes[1, 0].axhline(mean[0], color='g', linestyle='--', label='True mean')
axes[1, 0].set_title('MCMC Hammer Trace: Walker 0, Param 1')
axes[1, 0].set_xlabel('Iteration')
axes[1, 0].legend()

# Plot 5: Hammer Samples
burn_in_hammer = 50 * n_walkers
hammer_samples_burned = hammer_samples[burn_in_hammer:]
axes[1, 1].contour(X, Y, Z, levels=10, alpha=0.5)
axes[1, 1].scatter(hammer_samples_burned[:, 0], hammer_samples_burned[:, 1],
                   alpha=0.3, s=5, label='Hammer Samples')
axes[1, 1].plot(mean[0], mean[1], 'r*', markersize=15, label='True Mean')
axes[1, 1].set_title('MCMC Hammer Samples')
axes[1, 1].set_xlabel('Parameter 1')
axes[1, 1].set_ylabel('Parameter 2')
axes[1, 1].legend()

# Plot 6: Hammer Marginal distributions
axes[1, 2].hist(hammer_samples_burned[:, 0], bins=50, density=True, alpha=0.7, label='Samples')
axes[1, 2].plot(x_theory, y_theory, 'r-', linewidth=2, label='True PDF')
axes[1, 2].set_title('MCMC Hammer Marginal: Parameter 1')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nSummary Statistics:")
print(f"True Mean: {mean}")
print(f"MH Estimated Mean: {mh_samples_burned.mean(axis=0)}")
print(f"Hammer Estimated Mean: {hammer_samples_burned.mean(axis=0)}")
print(f"\nTrue Covariance:\n{cov}")
print(f"\nMH Estimated Covariance:\n{np.cov(mh_samples_burned.T)}")
print(f"\nHammer Estimated Covariance:\n{np.cov(hammer_samples_burned.T)}")

## 4. Probability Distributions

The package provides two implementations of probability distributions:
1. **JAX-accelerated distributions** (`Continuous`, `Discrete`): Using TensorFlow Probability backend with GPU support
2. **NumPy distributions** (`Probablity_distributions`): Pure NumPy implementations

Let's explore the JAX-accelerated continuous and discrete distributions.

In [ ]:
# Continuous Distributions
print("=== Continuous Distributions ===\n")

# Normal Distribution
normal = Normal(loc=0.0, scale=1.0)
x = jnp.linspace(-4, 4, 1000)
pdf = normal.prob(x)
cdf = normal.cdf(x)
samples = normal.sample(random.PRNGKey(0), sample_shape=(1000,))

print(f"Normal(0, 1):")
print(f"  Mean: {normal.mean():.4f}")
print(f"  Variance: {normal.variance():.4f}")
print(f"  Sample mean: {jnp.mean(samples):.4f}")
print(f"  Sample std: {jnp.std(samples):.4f}\n")

# Beta Distribution
beta = Beta(concentration1=2.0, concentration0=5.0)
x_beta = jnp.linspace(0, 1, 1000)
pdf_beta = beta.prob(x_beta)
samples_beta = beta.sample(random.PRNGKey(1), sample_shape=(1000,))

print(f"Beta(2, 5):")
print(f"  Mean: {beta.mean():.4f}")
print(f"  Variance: {beta.variance():.4f}")
print(f"  Sample mean: {jnp.mean(samples_beta):.4f}\n")

# Gamma Distribution
gamma = Gamma(concentration=2.0, rate=1.0)
x_gamma = jnp.linspace(0, 10, 1000)
pdf_gamma = gamma.prob(x_gamma)
samples_gamma = gamma.sample(random.PRNGKey(2), sample_shape=(1000,))

print(f"Gamma(2, 1):")
print(f"  Mean: {gamma.mean():.4f}")
print(f"  Variance: {gamma.variance():.4f}")
print(f"  Sample mean: {jnp.mean(samples_gamma):.4f}\n")

# Student's T Distribution
student_t = StudentT(df=3.0, loc=0.0, scale=1.0)
pdf_t = student_t.prob(x)
samples_t = student_t.sample(random.PRNGKey(3), sample_shape=(1000,))

print(f"StudentT(df=3):")
print(f"  Mean: {student_t.mean():.4f}")
print(f"  Variance: {student_t.variance():.4f}")
print(f"  Sample mean: {jnp.mean(samples_t):.4f}\n")

# Discrete Distributions
print("=== Discrete Distributions ===\n")

# Binomial Distribution
binomial = Binomial(total_count=10, probs=0.3)
k = jnp.arange(0, 11)
pmf_binomial = binomial.prob(k)
samples_binomial = binomial.sample(random.PRNGKey(4), sample_shape=(1000,))

print(f"Binomial(n=10, p=0.3):")
print(f"  Mean: {binomial.mean():.4f}")
print(f"  Variance: {binomial.variance():.4f}")
print(f"  Sample mean: {jnp.mean(samples_binomial):.4f}\n")

# Poisson Distribution
poisson = Poisson(rate=3.0)
k_poisson = jnp.arange(0, 15)
pmf_poisson = poisson.prob(k_poisson)
samples_poisson = poisson.sample(random.PRNGKey(5), sample_shape=(1000,))

print(f"Poisson(λ=3):")
print(f"  Mean: {poisson.mean():.4f}")
print(f"  Variance: {poisson.variance():.4f}")
print(f"  Sample mean: {jnp.mean(samples_poisson):.4f}")

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Normal PDF and CDF
axes[0, 0].plot(x, pdf, 'b-', linewidth=2, label='PDF')
axes[0, 0].fill_between(x, pdf, alpha=0.3)
axes[0, 0].set_title('Normal(0, 1) - PDF')
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('Density')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

ax_twin = axes[0, 0].twinx()
ax_twin.plot(x, cdf, 'r-', linewidth=2, alpha=0.7, label='CDF')
ax_twin.set_ylabel('Cumulative Probability', color='r')
ax_twin.tick_params(axis='y', labelcolor='r')

# Beta Distribution
axes[0, 1].plot(x_beta, pdf_beta, 'b-', linewidth=2)
axes[0, 1].fill_between(x_beta, pdf_beta, alpha=0.3)
axes[0, 1].hist(samples_beta, bins=30, density=True, alpha=0.5, color='orange', label='Samples')
axes[0, 1].set_title('Beta(2, 5)')
axes[0, 1].set_xlabel('x')
axes[0, 1].set_ylabel('Density')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# Gamma Distribution
axes[0, 2].plot(x_gamma, pdf_gamma, 'b-', linewidth=2)
axes[0, 2].fill_between(x_gamma, pdf_gamma, alpha=0.3)
axes[0, 2].hist(samples_gamma, bins=30, density=True, alpha=0.5, color='orange', label='Samples')
axes[0, 2].set_title('Gamma(2, 1)')
axes[0, 2].set_xlabel('x')
axes[0, 2].set_ylabel('Density')
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].legend()

# Student's T Distribution comparison with Normal
axes[1, 0].plot(x, pdf, 'b-', linewidth=2, label='Normal(0, 1)')
axes[1, 0].plot(x, pdf_t, 'r-', linewidth=2, label='StudentT(df=3)')
axes[1, 0].fill_between(x, pdf_t, alpha=0.3, color='red')
axes[1, 0].set_title('Student\'s T vs Normal')
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('Density')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# Binomial Distribution
axes[1, 1].bar(k, pmf_binomial, alpha=0.7, color='blue', label='PMF')
axes[1, 1].hist(samples_binomial, bins=k, density=True, alpha=0.5, color='orange', label='Samples')
axes[1, 1].set_title('Binomial(n=10, p=0.3)')
axes[1, 1].set_xlabel('k')
axes[1, 1].set_ylabel('Probability')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

# Poisson Distribution
axes[1, 2].bar(k_poisson, pmf_poisson, alpha=0.7, color='blue', label='PMF')
axes[1, 2].hist(samples_poisson, bins=k_poisson, density=True, alpha=0.5, color='orange', label='Samples')
axes[1, 2].set_title('Poisson(λ=3)')
axes[1, 2].set_xlabel('k')
axes[1, 2].set_ylabel('Probability')
axes[1, 2].grid(True, alpha=0.3)
axes[1, 2].legend()

plt.tight_layout()
plt.show()

## 5. Principal Component Analysis (PCA) and Probabilistic PCA

The package provides both standard PCA and Probabilistic PCA (PPCA) implementations:
- **PCA**: Classical eigenvalue decomposition for dimensionality reduction
- **PPCA**: Probabilistic interpretation with EM algorithm, handles missing data

Let's demonstrate both on synthetic data with missing values.

In [ ]:
# Generate synthetic data
np.random.seed(42)
n_samples = 500
n_features = 10
n_components = 3

# Create latent structure
true_loading = np.random.randn(n_features, n_components)
latent_vars = np.random.randn(n_samples, n_components)
X_true = latent_vars @ true_loading.T + np.random.randn(n_samples, n_features) * 0.5

# Create dataset with missing values
X_missing = X_true.copy()
missing_mask = np.random.rand(n_samples, n_features) < 0.2  # 20% missing
X_missing[missing_mask] = np.nan

print(f"Data shape: {X_missing.shape}")
print(f"Missing values: {np.isnan(X_missing).sum()} ({np.isnan(X_missing).mean()*100:.1f}%)")

# Standard PCA (requires complete data)
X_complete = X_true  # Use complete data for standard PCA
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_complete)

print(f"\n=== Standard PCA ===")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Cumulative variance: {np.cumsum(pca.explained_variance_ratio_)}")

# Probabilistic PCA (handles missing data)
ppca = PPCA(latent_dim=n_components, max_iter=50, tol=1e-4)
X_ppca, W_ppca, mu_ppca, sigma2_ppca = ppca.fit_transform(X_missing)

print(f"\n=== Probabilistic PCA ===")
print(f"Loading matrix shape: {W_ppca.shape}")
print(f"Mean vector shape: {mu_ppca.shape}")
print(f"Noise variance: {sigma2_ppca:.4f}")
print(f"Transformed data shape: {X_ppca.shape}")

# Reconstruct data
X_reconstructed_pca = X_pca @ pca.components_ + pca.mean_
X_reconstructed_ppca = X_ppca @ W_ppca.T + mu_ppca

# Calculate reconstruction errors (only on non-missing values)
valid_mask = ~np.isnan(X_missing)
mse_pca = np.mean((X_true[valid_mask] - X_reconstructed_pca[valid_mask])**2)
mse_ppca = np.mean((X_true[valid_mask] - X_reconstructed_ppca[valid_mask])**2)

print(f"\nReconstruction MSE:")
print(f"  PCA: {mse_pca:.6f}")
print(f"  PPCA: {mse_ppca:.6f}")

In [ ]:
# Visualize PCA results
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Plot 1: PCA projection (first 2 components)
scatter1 = axes[0, 0].scatter(X_pca[:, 0], X_pca[:, 1], c=latent_vars[:, 0], 
                               cmap='viridis', alpha=0.6, s=20)
axes[0, 0].set_title('PCA Projection (PC1 vs PC2)')
axes[0, 0].set_xlabel('PC1')
axes[0, 0].set_ylabel('PC2')
plt.colorbar(scatter1, ax=axes[0, 0], label='True Latent Var 1')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: PPCA projection (first 2 components)
scatter2 = axes[0, 1].scatter(X_ppca[:, 0], X_ppca[:, 1], c=latent_vars[:, 0],
                               cmap='viridis', alpha=0.6, s=20)
axes[0, 1].set_title('PPCA Projection (Comp 1 vs Comp 2)')
axes[0, 1].set_xlabel('Component 1')
axes[0, 1].set_ylabel('Component 2')
plt.colorbar(scatter2, ax=axes[0, 1], label='True Latent Var 1')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Explained variance
axes[0, 2].bar(range(1, n_components+1), pca.explained_variance_ratio_, 
               alpha=0.7, color='blue')
axes[0, 2].plot(range(1, n_components+1), np.cumsum(pca.explained_variance_ratio_), 
                'ro-', linewidth=2, label='Cumulative')
axes[0, 2].set_title('PCA Explained Variance')
axes[0, 2].set_xlabel('Component')
axes[0, 2].set_ylabel('Variance Ratio')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: Original vs reconstructed (PCA)
sample_idx = 0
axes[1, 0].plot(X_true[sample_idx], 'bo-', label='Original', linewidth=2)
axes[1, 0].plot(X_reconstructed_pca[sample_idx], 'r^-', label='PCA Reconstruction', linewidth=2)
axes[1, 0].set_title('PCA Reconstruction (Sample 0)')
axes[1, 0].set_xlabel('Feature')
axes[1, 0].set_ylabel('Value')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Original vs reconstructed (PPCA)
axes[1, 1].plot(X_true[sample_idx], 'bo-', label='Original', linewidth=2)
axes[1, 1].plot(X_reconstructed_ppca[sample_idx], 'g^-', label='PPCA Reconstruction', linewidth=2)
# Mark missing values
missing_idx = np.where(np.isnan(X_missing[sample_idx]))[0]
if len(missing_idx) > 0:
    axes[1, 1].scatter(missing_idx, X_true[sample_idx, missing_idx], 
                      color='red', s=100, marker='x', linewidths=3, label='Missing')
axes[1, 1].set_title('PPCA Reconstruction (Sample 0)')
axes[1, 1].set_xlabel('Feature')
axes[1, 1].set_ylabel('Value')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Plot 6: Reconstruction error distribution
errors_pca = (X_true - X_reconstructed_pca).flatten()
errors_ppca = (X_true - X_reconstructed_ppca).flatten()
axes[1, 2].hist(errors_pca, bins=50, alpha=0.5, label='PCA', density=True)
axes[1, 2].hist(errors_ppca, bins=50, alpha=0.5, label='PPCA', density=True)
axes[1, 2].set_title('Reconstruction Error Distribution')
axes[1, 2].set_xlabel('Error')
axes[1, 2].set_ylabel('Density')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Ordinary Differential Equation Solvers

The package provides multiple ODE solver methods for numerical integration:
- **Explicit methods**: Euler, RK2, RK3, RK4
- **Multi-step methods**: Adams-Bashforth (AB2-5), Adams-Bashforth-Adams-Moulton (ABAM2-5)

Let's solve the classic Lorenz system, a chaotic dynamical system:

$$
\begin{aligned}
\frac{dx}{dt} &= \sigma(y - x) \\
\frac{dy}{dt} &= x(\rho - z) - y \\
\frac{dz}{dt} &= xy - \beta z
\end{aligned}
$$

In [ ]:
# Define the Lorenz system
def lorenz_system(t, state, sigma=10.0, rho=28.0, beta=8.0/3.0):
    """Lorenz attractor differential equations"""
    x, y, z = state
    dx_dt = sigma * (y - x)
    dy_dt = x * (rho - z) - y
    dz_dt = x * y - beta * z
    return jnp.array([dx_dt, dy_dt, dz_dt])

# Initial conditions and time span
initial_state = jnp.array([1.0, 1.0, 1.0])
t_start = 0.0
t_end = 50.0
dt = 0.01
n_steps = int((t_end - t_start) / dt)

# Solve using different methods
print("Solving Lorenz system with different ODE methods...")
print(f"Time span: [{t_start}, {t_end}], dt={dt}, steps={n_steps}\n")

# Create solver instance
solver = ODESolvers()

# Solve with RK4 (most accurate explicit method)
print("Running RK4...")
t_rk4, solution_rk4 = solver.RK4(
    f=lorenz_system,
    y0=initial_state,
    t0=t_start,
    tf=t_end,
    N=n_steps
)

# Solve with RK2 (for comparison)
print("Running RK2...")
t_rk2, solution_rk2 = solver.RK2(
    f=lorenz_system,
    y0=initial_state,
    t0=t_start,
    tf=t_end,
    N=n_steps
)

# Solve with Euler (simplest method)
print("Running Euler...")
t_euler, solution_euler = solver.Euler(
    f=lorenz_system,
    y0=initial_state,
    t0=t_start,
    tf=t_end,
    N=n_steps
)

# Solve with Adams-Bashforth 4 (multi-step method)
print("Running Adams-Bashforth 4...")
t_ab4, solution_ab4 = solver.AdamsBashforth4(
    f=lorenz_system,
    y0=initial_state,
    t0=t_start,
    tf=t_end,
    N=n_steps
)

print("\n✅ All ODE solutions computed successfully!")

In [ ]:
# Visualize the Lorenz attractor
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(18, 12))

# 3D trajectory plot
ax1 = fig.add_subplot(2, 2, 1, projection='3d')
ax1.plot(solution_rk4[:, 0], solution_rk4[:, 1], solution_rk4[:, 2], 
         linewidth=0.5, alpha=0.8, color='blue')
ax1.scatter([initial_state[0]], [initial_state[1]], [initial_state[2]], 
            color='green', s=100, marker='o', label='Start')
ax1.scatter([solution_rk4[-1, 0]], [solution_rk4[-1, 1]], [solution_rk4[-1, 2]], 
            color='red', s=100, marker='x', label='End')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.set_title('Lorenz Attractor - 3D Trajectory (RK4)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Time series plots
ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(t_rk4, solution_rk4[:, 0], label='x(t)', linewidth=1, alpha=0.8)
ax2.plot(t_rk4, solution_rk4[:, 1], label='y(t)', linewidth=1, alpha=0.8)
ax2.plot(t_rk4, solution_rk4[:, 2], label='z(t)', linewidth=1, alpha=0.8)
ax2.set_xlabel('Time')
ax2.set_ylabel('State Variables')
ax2.set_title('Lorenz System - Time Series (RK4)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Comparison: Phase space (x vs z)
ax3 = fig.add_subplot(2, 2, 3)
ax3.plot(solution_rk4[:, 0], solution_rk4[:, 2], label='RK4', linewidth=1.5, alpha=0.9)
ax3.plot(solution_rk2[:, 0], solution_rk2[:, 2], label='RK2', linewidth=1.0, alpha=0.7, linestyle='--')
ax3.plot(solution_euler[:, 0], solution_euler[:, 2], label='Euler', linewidth=0.8, alpha=0.6, linestyle=':')
ax3.set_xlabel('x')
ax3.set_ylabel('z')
ax3.set_title('Phase Space Comparison (x vs z)')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Method comparison: difference from RK4
ax4 = fig.add_subplot(2, 2, 4)
error_rk2 = jnp.linalg.norm(solution_rk2 - solution_rk4, axis=1)
error_euler = jnp.linalg.norm(solution_euler - solution_rk4, axis=1)
error_ab4 = jnp.linalg.norm(solution_ab4 - solution_rk4, axis=1)

ax4.semilogy(t_rk2, error_rk2, label='RK2 vs RK4', linewidth=2)
ax4.semilogy(t_euler, error_euler, label='Euler vs RK4', linewidth=2)
ax4.semilogy(t_ab4, error_ab4, label='AB4 vs RK4', linewidth=2)
ax4.set_xlabel('Time')
ax4.set_ylabel('Error (L2 norm)')
ax4.set_title('Solution Error Relative to RK4')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nMethod Comparison Statistics:")
print(f"RK2 mean error:   {jnp.mean(error_rk2):.6f}")
print(f"Euler mean error: {jnp.mean(error_euler):.6f}")
print(f"AB4 mean error:   {jnp.mean(error_ab4):.6f}")

## 7. Advanced Example: Bayesian Linear Regression with MCMC

Let's combine multiple components to perform Bayesian linear regression:
- Generate synthetic data with noise
- Define a probabilistic model with priors
- Use MCMC to sample from the posterior distribution
- Visualize uncertainty in predictions

In [ ]:
# Generate synthetic linear regression data
np.random.seed(42)
n_data = 100
true_slope = 2.5
true_intercept = 1.5
true_noise_std = 0.5

x_data = np.linspace(0, 10, n_data)
y_data = true_slope * x_data + true_intercept + np.random.normal(0, true_noise_std, n_data)

# Define the Bayesian model
def linear_model(params, x):
    """Linear regression model: y = slope * x + intercept"""
    slope, intercept = params
    return slope * x + intercept

def log_posterior(params):
    """Log posterior probability: log P(params | data)"""
    slope, intercept = params
    
    # Priors (weakly informative)
    log_prior_slope = -0.5 * ((slope - 0) / 5.0)**2  # Normal(0, 5)
    log_prior_intercept = -0.5 * ((intercept - 0) / 5.0)**2  # Normal(0, 5)
    
    # Likelihood
    predictions = linear_model(params, x_data)
    residuals = y_data - predictions
    log_likelihood = -0.5 * np.sum((residuals / true_noise_std)**2)
    
    return log_prior_slope + log_prior_intercept + log_likelihood

# Run MCMC sampling
n_samples = 10000
n_burnin = 2000
initial_params = np.array([0.0, 0.0])

print("Running MCMC for Bayesian Linear Regression...")
print(f"True parameters: slope={true_slope}, intercept={true_intercept}")
print(f"Sampling {n_samples} iterations with {n_burnin} burn-in...\n")

# Metropolis-Hastings
mh_sampler = MetropolisHastings(
    initial_theta=initial_params,
    log_posterior=log_posterior,
    proposal_std=0.1
)

samples = []
for _ in tqdm(range(n_samples), desc="MCMC Sampling"):
    sample, _ = mh_sampler.sample()
    samples.append(sample)

samples = np.array(samples)[n_burnin:]

# Compute posterior statistics
slope_samples = samples[:, 0]
intercept_samples = samples[:, 1]

print(f"\nPosterior Estimates:")
print(f"Slope:     {slope_samples.mean():.3f} ± {slope_samples.std():.3f} (true: {true_slope})")
print(f"Intercept: {intercept_samples.mean():.3f} ± {intercept_samples.std():.3f} (true: {true_intercept})")
print(f"\nAcceptance rate: {mh_sampler.acceptance_rate:.2%}")

In [ ]:
# Visualize Bayesian Linear Regression results
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Plot 1: Data and posterior predictions
x_plot = np.linspace(0, 10, 200)
axes[0, 0].scatter(x_data, y_data, alpha=0.5, label='Data', s=20)

# Plot posterior predictive samples
for i in np.random.choice(len(samples), 100):
    y_pred = linear_model(samples[i], x_plot)
    axes[0, 0].plot(x_plot, y_pred, 'C1-', alpha=0.05)

# Plot mean prediction
mean_slope = slope_samples.mean()
mean_intercept = intercept_samples.mean()
y_mean = linear_model([mean_slope, mean_intercept], x_plot)
axes[0, 0].plot(x_plot, y_mean, 'r-', linewidth=3, label='Posterior Mean')

# Plot true line
y_true = linear_model([true_slope, true_intercept], x_plot)
axes[0, 0].plot(x_plot, y_true, 'g--', linewidth=2, label='True Model')

axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('y')
axes[0, 0].set_title('Bayesian Linear Regression')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Posterior distribution of slope
axes[0, 1].hist(slope_samples, bins=50, density=True, alpha=0.7, label='Posterior')
axes[0, 1].axvline(true_slope, color='g', linestyle='--', linewidth=2, label='True Value')
axes[0, 1].axvline(slope_samples.mean(), color='r', linestyle='-', linewidth=2, label='Posterior Mean')
axes[0, 1].set_xlabel('Slope')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title('Posterior Distribution: Slope')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Posterior distribution of intercept
axes[0, 2].hist(intercept_samples, bins=50, density=True, alpha=0.7, label='Posterior')
axes[0, 2].axvline(true_intercept, color='g', linestyle='--', linewidth=2, label='True Value')
axes[0, 2].axvline(intercept_samples.mean(), color='r', linestyle='-', linewidth=2, label='Posterior Mean')
axes[0, 2].set_xlabel('Intercept')
axes[0, 2].set_ylabel('Density')
axes[0, 2].set_title('Posterior Distribution: Intercept')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: Joint posterior
axes[1, 0].hexbin(slope_samples, intercept_samples, gridsize=30, cmap='Blues', alpha=0.7)
axes[1, 0].scatter([true_slope], [true_intercept], color='red', s=200, marker='*', 
                   edgecolors='black', linewidths=2, label='True Values', zorder=5)
axes[1, 0].scatter([mean_slope], [mean_intercept], color='orange', s=100, marker='o',
                   edgecolors='black', linewidths=2, label='Posterior Mean', zorder=5)
axes[1, 0].set_xlabel('Slope')
axes[1, 0].set_ylabel('Intercept')
axes[1, 0].set_title('Joint Posterior Distribution')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Trace plots
axes[1, 1].plot(slope_samples, alpha=0.7, linewidth=0.5)
axes[1, 1].axhline(true_slope, color='g', linestyle='--', linewidth=2, label='True Value')
axes[1, 1].set_xlabel('Iteration (after burn-in)')
axes[1, 1].set_ylabel('Slope')
axes[1, 1].set_title('Trace Plot: Slope')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(intercept_samples, alpha=0.7, linewidth=0.5)
axes[1, 2].axhline(true_intercept, color='g', linestyle='--', linewidth=2, label='True Value')
axes[1, 2].set_xlabel('Iteration (after burn-in)')
axes[1, 2].set_ylabel('Intercept')
axes[1, 2].set_title('Trace Plot: Intercept')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute credible intervals
slope_ci = np.percentile(slope_samples, [2.5, 97.5])
intercept_ci = np.percentile(intercept_samples, [2.5, 97.5])

print(f"\n95% Credible Intervals:")
print(f"Slope:     [{slope_ci[0]:.3f}, {slope_ci[1]:.3f}]")
print(f"Intercept: [{intercept_ci[0]:.3f}, {intercept_ci[1]:.3f}]")

## Summary

This tutorial demonstrated the core capabilities of the **Optimization and Probabilistic Modeling** package:

### Key Features Covered:

1. **Optimization Algorithms** - SGD, Adam, and RMSprop optimizers for gradient-based optimization
2. **MCMC Sampling** - Metropolis-Hastings and MCMC Hammer for Bayesian inference
3. **Probability Distributions** - JAX-accelerated continuous and discrete distributions
4. **Dimensionality Reduction** - PCA and PPCA with missing data support
5. **ODE Solvers** - Multiple numerical methods for solving dynamical systems
6. **Bayesian Modeling** - Complete workflow from model specification to posterior inference

### Next Steps:

- Explore other distributions in `Continuous.py` and `Discrete.py`
- Experiment with Factor Analysis and Canonical Correlation Analysis
- Try different ODE solver methods for your dynamical systems
- Implement custom likelihood functions for your specific models
- Use the `ModelParallelizer` for efficient parallel model evaluation

### Resources:

- **GitHub Repository**: [Optimization-and-probabilistic-modeling](https://github.com/rezaaskary/Optimization-and-probabilistic-modeling)
- **Documentation**: Check the docstrings in each module for detailed API information
- **Issues & Contributions**: Feel free to open issues or contribute on GitHub

---

**Thank you for using this package!** 🚀

For questions or feedback, please reach out through GitHub issues.